In [ ]:
from __future__ import annotations
import sys
import os
import json
import math
import logging
from pathlib import Path
from typing import Callable, Optional
from uuid import uuid4
from collections import defaultdict

In [ ]:
import cv2
import numpy as np
import torch
from torchvision import models, transforms
from PIL import Image

In [ ]:
import psycopg2
!pip install psycopg2-binary pgvector
from pgvector.psycopg2 import register_vector
from psycopg2.extras import execute_values
import kagglehub
import shutil

In [ ]:
path_descarga = kagglehub.dataset_download("gpiosenka/70-dog-breedsimage-data-set")
print("Descargado en:", path_descarga)

PATH_DESTINO = "/content/tuia-dog-recognition-app/data"

if os.path.exists(PATH_DESTINO):
    shutil.rmtree(PATH_DESTINO)


shutil.copytree(path_descarga, PATH_DESTINO)
print(f"Dataset copiado: {PATH_DESTINO}")

Using Colab cache for faster access to the '70-dog-breedsimage-data-set' dataset.
Descargado en: /kaggle/input/70-dog-breedsimage-data-set
Dataset copiado: /content/tuia-dog-recognition-app/data


In [ ]:
URL_REPO = "https://github.com/FacundoMol/tuia-dog-recognition-app"

!git clone {URL_REPO} /content/repo_temporal

%cd /content/tuia-dog-recognition-app
!git pull
%cd /content/tuia-dog-recognition-app
!git checkout desarrollo
!git pull origin desarrollo

fatal: destination path '/content/repo_temporal' already exists and is not an empty directory.
/content/tuia-dog-recognition-app
Already up to date.
/content/tuia-dog-recognition-app
D	data/dataset/.gitkeep
D	data/embeddings.json
Branch 'desarrollo' set up to track remote branch 'desarrollo' from 'origin'.
Switched to a new branch 'desarrollo'
From https://github.com/FacundoMol/tuia-dog-recognition-app
 * branch            desarrollo -> FETCH_HEAD
Already up to date.


In [ ]:
ORIGEN = "/content/repo_temporal"
DESTINO = "/content/tuia-dog-recognition-app"

if os.path.exists(ORIGEN):
    for elemento in os.listdir(ORIGEN):
        if elemento != "data":
            ruta_origen = os.path.join(ORIGEN, elemento)
            ruta_destino = os.path.join(DESTINO, elemento)

            if os.path.exists(ruta_destino):
                if os.path.isdir(ruta_destino):
                    shutil.rmtree(ruta_destino)
                else:
                    os.remove(ruta_destino)

            shutil.move(ruta_origen, ruta_destino)




In [ ]:
RUTA_SRC = "/content/tuia-dog-recognition-app/src"


if RUTA_SRC not in sys.path:
    sys.path.insert(0, RUTA_SRC)
else:

    sys.path.remove(RUTA_SRC)
    sys.path.insert(0, RUTA_SRC)

try:
    from lib.schemas import EmbeddingRecord, Neighbor, SearchResult
    from lib.storage.base import EmbeddingStoreProtocol
    print("cargados")
except ModuleNotFoundError as e:
    print(f"Error{e}")

logger = logging.getLogger(__name__)

cargados


In [ ]:
class SimilarityService:
    """Etapa 1: buscador de imagenes por similitud.

    Funciones a implementar por el estudiante:
      - extract_embedding(image)
      - search_similar_images(embedding, top_k)
      - predict_breed_from_neighbors(results)

    La orquestacion (search, index_image, persistencia y metricas de similitud)
    ya esta provista y no debe modificarse sin justificarlo en el informe.
    """

    def __init__(
        self,
        store: EmbeddingStoreProtocol,
        similarity_metric: str,
        similarity_threshold: float,
        top_k: int,
        image_size: int,
        model_name: str,
        url_resolver: Optional[Callable[[Path], Optional[str]]] = None,
    ) -> None:
        self.store = store
        self.similarity_metric = similarity_metric
        self.similarity_threshold = similarity_threshold
        self.top_k = top_k
        self.image_size = image_size
        self.model_name = model_name
        self.url_resolver = url_resolver

        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

        self.preprocess = transforms.Compose([
            transforms.Resize(256),
            transforms.CenterCrop(self.image_size),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        ])

        self.base_model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
        self.model = torch.nn.Sequential(*list(self.base_model.children())[:-1])
        self.model = self.model.to(self.device)
        self.model.eval()

In [ ]:
def _load_image(self, source_path: str) -> np.ndarray:
        image = cv2.imread(str(source_path))
        if image is None:
            raise ValueError(f"Could not read image: {source_path}")
        # BGR uint8 (convencion OpenCV)
        return image

In [ ]:
def extract_embedding(self, image: np.ndarray) -> list[float]:
        """
        Genera el embedding de una imagen usando un modelo pre-entrenado en
        ImageNet (ej: ResNet50, EfficientNet, ConvNeXt) sin la capa de
        clasificacion final.
        """
        image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        pil_img = Image.fromarray(image_rgb)
        tensor_img = self.preprocess(pil_img).unsqueeze(0).to(self.device)

        with torch.no_grad():
            output = self.model(tensor_img)
            embedding_tensor = torch.flatten(output).cpu()
            embedding_list = embedding_tensor.numpy().tolist()

        return embedding_list

In [ ]:
def poblar_vector_db(self, image_records: list[dict], batch_size: int = 32):
        """Puebla la base de datos de pgvector, con batching de 32: id_imagen, embedding, path, breed y metadata"""
        conn = psycopg2.connect(
            host=getattr(self, "db_host", "localhost"),
            database=getattr(self, "db_name", "tu_base_datos"),
            user=getattr(self, "db_user", "postgres"),
            password=getattr(self, "db_password", "password"),
            port=getattr(self, "db_port", 5432)
        )

        cursor = conn.cursor()
        register_vector(conn)

        cursor.execute("CREATE EXTENSION IF NOT EXISTS vector;")
        cursor.execute("""CREATE TABLE IF NOT EXISTS dog_embeddings(
                id_imagen TEXT PRIMARY KEY,
                path TEXT NOT NULL UNIQUE,
                breed TEXT NOT NULL,
                metadata JSONB NOT NULL,
                embedding vector(2048));""")
        conn.commit()

        print(f"carga de {len(image_records)} registros vectoriales")

        for i in range(0, len(image_records), batch_size):
            sub_list = image_records[i : i + batch_size]
            valid_tensors = []
            valid_metadata = []

            for record in sub_list:
                path = record['path']
                breed = record['breed']
                meta_dict = record.get('metadata', {})

                if not os.path.exists(path):
                    continue

                image = cv2.imread(path)
                if image is None:
                    continue

                image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
                pil_img = Image.fromarray(image_rgb)
                tensor_img = self.preprocess(pil_img)

                valid_tensors.append(tensor_img)
                valid_metadata.append((path, breed, meta_dict))

            if not valid_tensors:
                continue

            try:
                batch_tensor = torch.stack(valid_tensors).to(self.device)

                with torch.no_grad():
                    outputs = self.model(batch_tensor)
                    if len(outputs.shape) > 2:
                        outputs = torch.flatten(outputs, start_dim=1)
                    embeddings_numpy = outputs.cpu().numpy()

                insert_data = []
                for idx, (path, breed, meta_dict) in enumerate(valid_metadata):
                    emb_list = embeddings_numpy[idx].tolist()
                    id_imagen = str(uuid4())
                    meta_json = json.dumps(meta_dict)
                    insert_data.append((id_imagen, path, breed, meta_json, emb_list))

                query = """INSERT INTO dog_embeddings (id_imagen, path, breed, metadata, embedding)
                    VALUES %s
                    ON CONFLICT (path)
                    DO UPDATE SET embedding = EXCLUDED.embedding, breed = EXCLUDED.breed, metadata = EXCLUDED.metadata;"""
                execute_values(cursor, query, insert_data)
                conn.commit()

                print(f"Lote procesado {min(i + batch_size, len(image_records))}/{len(image_records)}")

            except Exception as e:
                print(f"Error al procesar el lote, índice {i}: {str(e)}")
                conn.rollback()
                continue

        cursor.close()
        conn.close()
        print("Base de datos generada")

In [ ]:
def search_similar_images(self, embedding: list[float], top_k: int) -> list[Neighbor]:
        """Recupera de la base vectorial las top_k imagenes mas similares."""
        raw_results = self.store.search(embedding, top_k)
        neighbors = []

        for item in raw_results:
            path = getattr(item, 'path', item.get('path') if isinstance(item, dict) else item[1])
            breed = getattr(item, 'breed', item.get('breed') if isinstance(item, dict) else item[2])
            ref_embedding = getattr(item, 'embedding', item.get('embedding') if isinstance(item, dict) else item[4])

            if hasattr(ref_embedding, 'tolist'):
                ref_embedding = ref_embedding.tolist()

            score = self.similarity(embedding, ref_embedding)
            neighbor = Neighbor(path=path, breed=breed, score=score)
            neighbors.append(neighbor)

        neighbors.sort(key=lambda x: x.score, reverse=True)
        return neighbors

In [ ]:
def predict_breed_from_neighbors(self, results: list[Neighbor]) -> tuple[str, float]:
        """Predice la raza a partir de los vecinos recuperados."""
        if not results:
            return "unknown", 0.0

        best_score = results[0].score
        threshold = getattr(self, "similarity_threshold", 0.0)

        if best_score < threshold:
            return "unknown", float(best_score)

        breed_votes = defaultdict(float)
        for neighbor in results:
            breed_votes[neighbor.breed] += neighbor.score

        predicted_breed = max(breed_votes, key=breed_votes.get)
        return predicted_breed, float(best_score)

In [ ]:
def _cosine(self, a: np.ndarray, b: np.ndarray) -> float:
        denom = np.linalg.norm(a) * np.linalg.norm(b)
        if denom == 0:
            return 0.0
        return float(np.dot(a, b) / denom)

def _l2_similarity(self, a: np.ndarray, b: np.ndarray) -> float:
        dist = float(np.linalg.norm(a - b))
        return 1.0 / (1.0 + dist)

def similarity(self, query: list[float], ref: list[float]) -> float:
        a = np.asarray(query, dtype=np.float32)
        b = np.asarray(ref, dtype=np.float32)
        if self.similarity_metric.lower() == "l2":
            return self._l2_similarity(a, b)
        return self._cosine(a, b)

In [ ]:
def index_image(
        self, image_path: str, breed: str, metadata: dict[str, object] | None = None
    ) -> EmbeddingRecord:
        """Extrae el embedding de una imagen del dataset y lo persiste en la base vectorial."""
        image = self._load_image(image_path)
        embedding = self.extract_embedding(image)
        record = EmbeddingRecord(
            id_imagen=str(uuid4()),
            embedding=embedding,
            path=str(image_path),
            breed=breed,
            metadata=metadata or {},
        )
        self.store.append(record)
        return record

def _with_url(self, neighbor: Neighbor) -> Neighbor:
        if self.url_resolver is not None and not neighbor.url:
            neighbor.url = self.url_resolver(Path(neighbor.path))
        return neighbor

def search(
        self,
        source_path: str,
        output_path: Path,
        embedding_fn: Optional[Callable[[np.ndarray], list[float]]] = None,
        model_name: Optional[str] = None,
        top_k: Optional[int] = None,
    ) -> str:
        """Pipeline completo de la Etapa 1: embedding -> vecinos -> raza predicha."""
        image = self._load_image(source_path)
        extractor = embedding_fn or self.extract_embedding
        embedding = extractor(image)

        k = int(top_k) if top_k else self.top_k
        neighbors = [self._with_url(n) for n in self.search_similar_images(embedding, k)]
        breed, score = self.predict_breed_from_neighbors(neighbors)
        logger.info("Predicted breed: %s (score=%.4f) for %s", breed, score, source_path)

        payload = SearchResult(
            source_path=source_path,
            model=model_name or self.model_name,
            predicted_breed=breed,
            score=round(float(score), 4),
            neighbors=neighbors,
        )
        output_path.mkdir(parents=True, exist_ok=True)
        result_file = output_path / f"result-{uuid4()}.json"
        result_file.write_text(
            json.dumps(payload.model_dump(), ensure_ascii=True, indent=2),
            encoding="utf-8",
        )
        return str(result_file)

In [ ]:

!apt-get update -qq
!apt-get install -y postgresql postgresql-contrib build-essential git postgresql-server-dev-all -qq

!git clone https://github.com/pgvector/pgvector.git /tmp/pgvector
%cd /tmp/pgvector
!make
!make install

%cd /content/tuia-dog-recognition-app

!pip install -q "psycopg[binary]"
!service postgresql start

!sudo -u postgres psql -c "ALTER USER postgres PASSWORD 'password';"
!sudo -u postgres psql -c "CREATE DATABASE tu_base_datos;"

print("\nPostgreSQL y pgvector se compilaron e iniciaron")
from lib.storage.pgvector_store import PgVectorEmbeddingStore
from lib.services.similarity_service import SimilarityService

store = PgVectorEmbeddingStore(host="localhost",
    port=5432,
    dbname="tu_base_datos",
    user="postgres",
    password="password",
    embedding_dim=2048)

service = SimilarityService(store=store,
    similarity_metric='cosine',
    similarity_threshold=0.5,
    top_k=5,
    image_size=224,
    model_name='resnet50')

print("Store instanciado")

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
fatal: destination path '/tmp/pgvector' already exists and is not an empty directory.
/tmp/pgvector
make: Nothing to be done for 'all'.
/bin/mkdir -p '/usr/lib/postgresql/14/lib'
/bin/mkdir -p '/usr/share/postgresql/14/extension'
/bin/mkdir -p '/usr/share/postgresql/14/extension'
/usr/bin/install -c -m 755  vector.so '/usr/lib/postgresql/14/lib/vector.so'
/usr/bin/install -c -m 644 .//vector.control '/usr/share/postgresql/14/extension/'
/usr/bin/install -c -m 644 .//sql/vector--0.1.0--0.1.1.sql .//sql/vector--0.1.1--0.1.3.sql .//sql/vector--0.1.3--0.1.4.sql .//sql/vector--0.1.4--0.1.5.sql .//sql/vector--0.1.5--0.1.6.sql .//sql/vector--0.1.6--0.1.7.sql .//sql/vector--0.1.7--0.1.8.sql .//sql/vector--0.1.8--0.2.0.sql .//sql/vector--0.2.0--0.2.1.sql .//sql/vector--0.2.1--0.2.2.sql .//sql/vector--0.2.2--0

100%|██████████| 97.8M/97.8M [00:00<00:00, 127MB/s]


Store instanciado


In [ ]:
carpeta_imagenes = "/content/tuia-dog-recognition-app/data/train"

image_records = []

if not os.path.exists(carpeta_imagenes):
    print(f"No se encontró la carpeta '{carpeta_imagenes}'")
else:
    for root, dirs, files in os.walk(carpeta_imagenes):
        for archivo in files:
            if archivo.lower().endswith(('.jpg', '.jpeg', '.png')):

                ruta_completa = os.path.join(root, archivo)
                raza = os.path.basename(root)
                image_records.append({"path": ruta_completa,"breed": raza})

    print(f"Se cargaron {len(image_records)} imágenes listas para indexar.")

Se cargaron 7946 imágenes listas para indexar.


In [ ]:
from lib.schemas import EmbeddingRecord

print(f"procesamiento e indexación en Postgres para {len(image_records)} imágenes")

device = getattr(service, 'device', 'cpu')
service.model.eval()

contador_exitos = 0

for idx, record in enumerate(image_records):
    try:
        ruta_str = record['path']
        raza_img = record['breed']

        try:
            img_input = Image.open(ruta_str).convert('RGB')
        except Exception:
            img_input = Path(ruta_str)

        img_tensors = service.preprocess(img_input)

        if isinstance(img_tensors, torch.Tensor):
            img_tensors = img_tensors.to(device)
            if img_tensors.ndim == 3:
                img_tensors = img_tensors.unsqueeze(0)

        with torch.no_grad():
            embedding_tensor = service.model(img_tensors)
            embedding_vector = embedding_tensor.squeeze().cpu().tolist()

        obj = EmbeddingRecord(
            id_imagen=str(idx),
            path=ruta_str,
            breed=raza_img,
            embedding=embedding_vector
        )

        service.store.append(obj)
        contador_exitos += 1

        if (idx + 1) % 800 == 0:
            print(f"Procesadas e insertadas {idx + 1}/{len(image_records)} imágenes")

    except Exception as e:
        print(f"error en índice {idx} (Ruta: {record.get('path')}): {e}")
        break

if contador_exitos == len(image_records):
    print(f"\n{len(image_records)} imágenes procesadas e indexadas")
else:
    print(f"\nse interrumpió. Se lograron indexar {contador_exitos} de {len(image_records)} imágenes")

procesamiento e indexación en Postgres para 7946 imágenes
Procesadas e insertadas 800/7946 imágenes
Procesadas e insertadas 1600/7946 imágenes
Procesadas e insertadas 2400/7946 imágenes
Procesadas e insertadas 3200/7946 imágenes
Procesadas e insertadas 4000/7946 imágenes
Procesadas e insertadas 4800/7946 imágenes
Procesadas e insertadas 5600/7946 imágenes
Procesadas e insertadas 6400/7946 imágenes
Procesadas e insertadas 7200/7946 imágenes

7946 imágenes procesadas e indexadas


In [ ]:
imagen_test_path = image_records[0]['path']
raza_real = image_records[0]['breed']
print(f"📸 Imagen de consulta: {imagen_test_path} (Raza Real: {raza_real})\n")

img_pil = Image.open(imagen_test_path).convert('RGB')
img_tensors = service.preprocess(img_pil).to(service.device).unsqueeze(0)

with torch.no_grad():
    embedding_query = service.model(img_tensors).squeeze().cpu().tolist()

vecinos = service.search_similar_images(embedding_query, top_k=10)

raza_predicha, score_obtenido = service.predict_breed_from_neighbors(vecinos)

print("RESULTADOS")
print(f"Raza Predicha por K-NN: {raza_predicha}")
print(f"Score/distancia mejor vecino: {score_obtenido}\n")

print(f"10 vecinos más cercanos en Postgres:")
for i, v in enumerate(vecinos):
    print(f"  {i+1}️⃣ Raza: {v.breed} | Score: {v.score:.4f} | Path: {v.path}")



📸 Imagen de consulta: /content/tuia-dog-recognition-app/data/train/Komondor/005.jpg (Raza Real: Komondor)

🎯 --- RESULTADOS DEL PIPELINE ---
🐶 Raza Predicha por K-NN: Komondor
📊 Score/Distancia del mejor vecino: 1.0

📋 Lista de los 10 vecinos más cercanos en Postgres:
  1️⃣ Raza: Komondor | Score: 1.0000 | Path: /content/tuia-dog-recognition-app/data/train/Komondor/005.jpg
  2️⃣ Raza: Komondor | Score: 0.9154 | Path: /content/tuia-dog-recognition-app/data/train/Komondor/054.jpg
  3️⃣ Raza: Komondor | Score: 0.9038 | Path: /content/tuia-dog-recognition-app/data/train/Komondor/066.jpg
  4️⃣ Raza: Komondor | Score: 0.8885 | Path: /content/tuia-dog-recognition-app/data/train/Komondor/075.jpg
  5️⃣ Raza: Komondor | Score: 0.8878 | Path: /content/tuia-dog-recognition-app/data/train/Komondor/044.jpg
  6️⃣ Raza: Komondor | Score: 0.8865 | Path: /content/tuia-dog-recognition-app/data/train/Komondor/014.jpg
  7️⃣ Raza: Komondor | Score: 0.8803 | Path: /content/tuia-dog-recognition-app/data/train

In [ ]:
correctas = 0
total = 0
suma_ndcg_10 = 0.0

with torch.no_grad():
    for idx, record in enumerate(image_records):
        try:
            ruta_str = record['path']
            raza_real = record['breed']

            img_pil = Image.open(ruta_str).convert('RGB')
            img_tensors = service.preprocess(img_pil).to(device).unsqueeze(0)

            embedding_tensor = service.model(img_tensors)
            embedding_query = embedding_tensor.squeeze().cpu().tolist()

            k_vecinos = getattr(service, 'top_k', 10)
            vecinos = service.search_similar_images(embedding_query, top_k=k_vecinos)

            raza_predicha, _ = service.predict_breed_from_neighbors(vecinos)


            razas_vecinos = [v.breed if hasattr(v, 'breed') else v.get('breed')
                for v in vecinos[:10]]

            relevancias = [1 if raza == raza_real else 0 for raza in razas_vecinos]

            ndcg_query = 0.0
            if sum(relevancias) > 0:
                dcg = sum(rel / math.log2(i + 2) for i, rel in enumerate(relevancias))

                relevancias_ideales = sorted(relevancias, reverse=True)
                idcg = sum(rel / math.log2(i + 2) for i, rel in enumerate(relevancias_ideales))

                ndcg_query = dcg / idcg if idcg > 0 else 0.0

            suma_ndcg_10 += ndcg_query

            total += 1
            if raza_predicha == raza_real:
                correctas += 1

            if (idx + 1) % 800 == 0:
                accuracy_actual = (correctas / total) * 100
                ndcg_actual = (suma_ndcg_10 / total) * 100
                print(f"Evaluadas {idx + 1}/{len(image_records)} | Accuracy: {accuracy_actual:.2f}% | Mean NDCG@10: {ndcg_actual:.2f}%")

        except Exception as e:
            print(f"Error procesando índice {idx} ({ruta_str}): {e}")
            continue


if total > 0:
    accuracy_final = (correctas / total) * 100
    ndcg_final = (suma_ndcg_10 / total) * 100
    print("\n" + "="*40)
    print(f"Total de imágenes evaluadas: {total}")
    print(f"Predicciones correctas: {correctas}")
    print(f"Accuracy modelo: {accuracy_final:.2f}%")
    print(f"Mean NDCG@10: {ndcg_final:.2f}%")
    print("="*40)
else:
    print("error")

Evaluadas 800/7946 | Accuracy: 93.12% | Mean NDCG@10: 98.96%
Evaluadas 1600/7946 | Accuracy: 93.12% | Mean NDCG@10: 98.91%
Evaluadas 2400/7946 | Accuracy: 94.21% | Mean NDCG@10: 98.91%
Evaluadas 3200/7946 | Accuracy: 94.19% | Mean NDCG@10: 98.95%
Evaluadas 4000/7946 | Accuracy: 94.80% | Mean NDCG@10: 99.02%
Evaluadas 4800/7946 | Accuracy: 94.90% | Mean NDCG@10: 99.09%
Evaluadas 5600/7946 | Accuracy: 94.96% | Mean NDCG@10: 99.13%
Evaluadas 6400/7946 | Accuracy: 95.20% | Mean NDCG@10: 99.15%
Evaluadas 7200/7946 | Accuracy: 94.93% | Mean NDCG@10: 99.11%

Total de imágenes evaluadas: 7946
Predicciones correctas: 7531
Accuracy modelo: 94.78%
Mean NDCG@10: 99.10%
